# ts_regress

> 对应代码：`EconometricsCode/code_in_notes/Chap.27/ts_regress.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.27`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.27")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

读入季度 GDP 数据，按时间变量与投资数据做一对一合并，再 `tsset` 声明时间序列结构。

In [ ]:
%%stata
clear
set more off

use ../datasets/quarterlyGDP.dta
merge 1:1 time using ../datasets/quarterlyInvest.dta

tsset time

构造 GDP 增长率：与去年同期（滞后 4 个季度）相比的增长率 $(gdp_t-gdp_{t-4})/gdp_{t-4}\times 100$，即同比增长率，这样得到的序列通常更接近平稳。

In [ ]:
%%stata
gen gdp_growth=(gdp-L4.gdp)/L4.gdp*100

下面用 Newey–West 回归考察投资（einvq0105）对 GDP 增长率的动态影响：从只含当期投资开始，逐步加入 1 至 4 阶滞后。由于时间序列回归的误差通常存在自相关，我们用 `newey ..., lag(4)` 计算 HAC 标准误（滞后截断取 4）。`L(0/k).x` 是 Stata 时间序列算子写法，表示当期到第 k 阶滞后。比较各方程中投资各期系数的大小与显著性，即可刻画投资对增长的动态乘数效应。

In [ ]:
%%stata
newey gdp_growth einvq0105, lag(4)
newey gdp_growth L(0/1).einvq0105, lag(4)
newey gdp_growth L(0/2).einvq0105, lag(4)
newey gdp_growth L(0/3).einvq0105, lag(4)
newey gdp_growth L(0/4).einvq0105, lag(4)